```
## Notebook: GraphSAGE_Data001.01_superfund_nodes_v1.ipynb
---  
#### Purpose
- Exclude nonfeature columns
- Clean raw superfund data
- Explode suboptimal string columns ['cat_code', cas_number'] to feature matrices
- Encode where required
- Create ready-to-use superfund nodes

#### Input
- GNN328_county_nodes.csv (raw county data)
- GNN328_super_feats.csv (1718 observations, 24 columns, 1 site_id per superfund site,   
  ['cat_code'] = delimited string of 1 - several values,   
  [cas_number'] = list of strings of 1 - several values)
- GNN328_super_gdf.gpkg

#### Processing

#### Output
- superfund_feature_matrix.csv
- superfund_nodes.csv
- superfund_edge_index.csv

#### Goal: Superfund nodes, county nodes, river nodes, modeling environmental hazards to analyze impact on cancer rate.
```

```
# ───────────────────────────────────────────────────────────────────────
# CODE CELL 1.0: ENVIRONMENT AND RUNTIME CONTROL
# ───────────────────────────────────────────────────────────────────────

Note: This notebook is being developed with Python in a Colab notebook using an L4 or A100 hardware accelerator. There are runtime-specific dependencies that must be aligned. Cell 1 should be run prior to starting work in each new runtime. After Cell 1 completes, the runtime must be restarted. Cell 1 may then be commented out to avoid reinstalling the dependencies after subsequent runtime restarts.
```

In [ ]:
# !pip uninstall -y torch torchvision torchaudio torch-scatter torch-sparse torch-geometric
# !pip install torch==2.0.0+cu118 torchvision==0.15.1+cu118 torchaudio==2.0.1+cu118 -f https://download.pytorch.org/whl/torch_stable.html
# !pip install torch-scatter torch-sparse torch-geometric -f https://data.pyg.org/whl/torch-2.0.0+cu118.html
# !pip install numpy==1.24.4
# !pip install Optuna

```
# ───────────────────────────────────────────────────────────────────────
# CODE CELL 2.0: IMPORT STATEMENTS
# ───────────────────────────────────────────────────────────────────────
```

In [ ]:
# Torch and torch utilities are for construction of HeteroGRAPH object
# ───────────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch_geometric
from torch_geometric.data import HeteroData
from torch.optim import Adam
from torch_geometric.nn import SAGEConv, HeteroConv
import numpy as np


# Sklearn encoder, training and evaluation tools
# ───────────────────────────────────────────────────────────────────────
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import optuna

# Data handling libraries
# ───────────────────────────────────────────────────────────────────────
from shapely.geometry import Point # For reading geometry in super_gdf
import geopandas as gpd            # For loading super_gdf
import pandas as pd

# Utilities
# ───────────────────────────────────────────────────────────────────────
from google.colab import files
import random
import inspect
import math
import time
import re

# Utilities
# ───────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import tabulate
from tabulate import tabulate
import collections
from collections import Counter

# Set random state for everything, everywhere, all at once.
# ───────────────────────────────────────────────────────────────────────
def jenny_setter(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

jenny = 8675309
jenny_setter(seed=jenny)

# Check runtime dependency control. Confirm correct PyTorch, CUDA, numpy.
# ───────────────────────────────────────────────────────────────────────
rows = [
        ["Torch", "Torch 2.0.0", torch.__version__],
        ["CUDA_version", "11.8", torch.version.cuda],
        ["NumPy", "NumPy 1.24.4", np.__version__]
       ]
print(tabulate(rows, headers=["Library", "Expected Version", "Current Version"], tablefmt="grid"))

+--------------+--------------------+-------------------+
| Library      | Expected Version   | Current Version   |
+==============+====================+===================+
| Torch        | Torch 2.0.0        | 2.0.0+cu118       |
+--------------+--------------------+-------------------+
| CUDA_version | 11.8               | 11.8              |
+--------------+--------------------+-------------------+
| NumPy        | NumPy 1.24.4       | 1.24.4            |
+--------------+--------------------+-------------------+


```
# ───────────────────────────────────────────────────────────────────────
# CELL 3.0: LOAD DATA, PREPROCESSING
# ───────────────────────────────────────────────────────────────────────
```

In [ ]:
# super_g is the superfund geopackage in long format, use: map geodata (fips)
# sfeats_df is the superfund data in wide format, use: base data for node construction.
# ──────────────────────────────────────────────────────────────────────────────
super_g = gpd.read_file('/content/GNN328_super_gdf.gpkg')
sfeats_df = pd.read_csv('/content/GNN328_super_feats.csv', low_memory=False, index_col=None)

# >>> Preparing super_g <<<
# ──────────────────────────────────────────────────────────────────────────────
# RangeIndex: 47143 entries, 0 to 47142 (long format, 1 observation each
# cas_number) When loaded from .csv site_id may enter as an int, float or object.
# It is a categorical site identifier, so ensure it's a string and pad to 7.
# ──────────────────────────────────────────────────────────────────────────────
super_g['site_id'] = super_g['site_id'].astype(str).str.zfill(7)

# ──────────────────────────────────────────────────────────────────────────────
# >>> Preparing sfeats_df <<<
# RangeIndex: 1718 entries, 0 to 1717 (wide format, 1 observation each site_id)
# sfeats_df.isna().sum() == 0.
# sfeats_df loaded from .csv likely requires type control and padding on site_id
# as well.
# Drop columns not needed for final feature matrix:
# ['region','site_name','state','zip_code','county','tribe',
#  'site_cat', 'reference_date','final_list_date','deleted_date']
# ───────────────────────────────────────────────────────────────────────
sfeats_df = sfeats_df.drop(columns=['region','site_name','state','zip_code',
                                    'county','tribe', 'site_cat', 'reference_date',
                                    'final_list_date','deleted_date'], axis=1)
sfeats_df['site_id'] = sfeats_df['site_id'].astype(str).str.zfill(7)
sfeats_df.dropna(subset=['hazard_rank_score'], inplace=True)

# ──────────────────────────────────────────────────────────────────────────────
# >>> Add fips to sfeats_df by mapping from super_g <<<
# We need to get fips mapped to correct observations in
# sfeats_df. So MERGE fips into sfeats_df FROM super_g.
# Slice a column pair super_g[['site_id', 'fips']], and
# merge into sfeats_df by site_id. Re-convert to string
# and re-pad to 7 to shut down type conversion issue.
# ──────────────────────────────────────────────────────────────────────────────
fips_lookup = super_g[['site_id', 'fips']].drop_duplicates()
sfeats_df = sfeats_df.merge(fips_lookup, on='site_id', how='left')
sfeats_df['site_id'] = sfeats_df['site_id'].astype(str).str.zfill(7)

# ──────────────────────────────────────────────────────────────────────────────
# >>> VALIDATION                                                             <<<
# >>> Length, number of unique fips and unique site_ids should be equal.     <<<
# ──────────────────────────────────────────────────────────────────────────────
print('──────────────────────────────────────')
print('sfeats_df length:      ', len(sfeats_df))
print('sfeats unique fips:    ', sfeats_df['site_id'].value_counts().sum())
print('sfeats unique site_id: ', sfeats_df['site_id'].value_counts().sum())
print('___All of the above should be equal___')
print('──────────────────────────────────────')
print()
print('sfeats_df columns: ')
display(sfeats_df.columns)
print('──────────────────────────────────────')
# print()
# print('── sfeats_df fips construction ───────')
# print("Superfund fips values (ensure five digits at upper and lower bounds: ")
# print(sfeats_df['fips'].sort_values())
# print('──────────────────────────────────────')
# print('── sfeats_df column types and info────')
# print('Superfund column types and info: ')
# print(sfeats_df['site_id'].value_counts())
# print('──────────────────────────────────────')

──────────────────────────────────────
sfeats_df length:       1606
sfeats unique fips:     1606
sfeats unique site_id:  1606
___All of the above should be equal___
──────────────────────────────────────

sfeats_df columns: 


Index(['site_id', 'lat', 'lon', 'fed_facility', 'native_land', 'npl_status',
       'cat_code', 'hazard_rank_score', 'final_list_days', 'proposed_days',
       'proposal_gulf', 'is_finalized', 'is_deleted', 'cas_number', 'fips'],
      dtype='object')

──────────────────────────────────────


```
# ───────────────────────────────────────────────────────────────────────
# CELL 4.0: SUPERFUND FEATURE SELECTION / CREATION
# ───────────────────────────────────────────────────────────────────────
```

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 4.1: ['cat_code'] MATRIX EXPLOSION
# ──────────────────────────────────────────────────────────────────────────────

# ──────────────────────────────────────────────────────────────────────────────
# >>> Create a list of unique values in ['cat_code']                         <<<
# ──────────────────────────────────────────────────────────────────────────────
cat_cols = ['mp', 're', 'wm', 'ot', 'mi', 'mt']

# ──────────────────────────────────────────────────────────────────────────────
# >>> Define a counter function to crawl the column and produce a count of   <<<
# >>> occurrences for each of the unique values. Returned dataframe is       <<<
# >>> indexed so that each row in the count matrix returned is aligned       <<<
# ──────────────────────────────────────────────────────────────────────────────
def count_per_string_max(series, cats):
    counts = series.fillna('').apply(
        lambda x: Counter(i.strip().lower() for i in x.split(';') if i.strip().lower() in cats)
    )
    return pd.DataFrame.from_records(counts, index=series.index).fillna(0).astype(int)

# ──────────────────────────────────────────────────────────────────────────────
# >>> Call the count_per_string function with arguments cat_code, cat_cols   <<<
# >>> assign site_ids from sfeats to cat_df to use as merge id. Group by     <<<
# >>> cat_df. We speculate a count of designations is useful; add sum column <<<
# ──────────────────────────────────────────────────────────────────────────────
cat_df = count_per_string_max(sfeats_df['cat_code'], cat_cols)
cat_df['site_id'] = sfeats_df['site_id']
site_cat_matrix = cat_df.groupby('site_id')[cat_cols].max()
site_cat_matrix['site_cat_sum'] = site_cat_matrix[['mp', 're', 'wm', 'ot', 'mi', 'mt']].sum(axis=1)

# ──────────────────────────────────────────────────────────────────────────────
# >>> VALIDATION OUTPUT                                                      <<<
# ──────────────────────────────────────────────────────────────────────────────
print('__cat_df length should equal sfeats_df length______')
print('cat_df length:    ', len(cat_df))
print('sfeats_df length: ', len(sfeats_df))
print('───────────────────────────────────────────────────')

# ──────────────────────────────────────────────────────────────────────────────
# >>> Merge the exploded ['cat_code'] back into sfeats_df                    <<<
# ──────────────────────────────────────────────────────────────────────────────
sfeats_df['site_id'] = sfeats_df['site_id'].astype(str)
sfeats_df = sfeats_df.merge(site_cat_matrix, on='site_id', how='left')



# ──────────────────────────────────────────────────────────────────────────────
# >>> VALIDATION OUTPUT                                                      <<<
# ──────────────────────────────────────────────────────────────────────────────
print('__Values appearing in sfeats_df should be counted__')
print('__accurately in the head/tail of site_cat_matrix___')
print('__sfeats_df[cat_code]______________________________')
display(sfeats_df['cat_code'])
print('───────────────────────────────────────────────────')
print()
print('site_cat_matrix head__')
display(site_cat_matrix.head(15))
print('───────────────────────────────────────────────────')
print()
print('site_cat_matrix tail__')
display(site_cat_matrix.tail(15))
print('───────────────────────────────────────────────────')
print()
print('__sfeats_df category matrix values should be same__')
print('__as the category matrix values in site_cat_matrix_')
print('__sfeats_df[cat_code]______________________________')
display(sfeats_df[['site_id', 'mp', 're', 'wm', 'ot', 'mi', 'mt', 'site_cat_sum']].head(15))
display(sfeats_df[['site_id', 'mp', 're', 'wm', 'ot', 'mi', 'mt', 'site_cat_sum']].tail(15))

# Remove cat_code: once validation checks are passed we no
# longer need it, it has been replaced with binary column.
sfeats_df = sfeats_df.drop(columns=['cat_code'], axis=1)

__cat_df length should equal sfeats_df length______
cat_df length:     1606
sfeats_df length:  1606
───────────────────────────────────────────────────
__Values appearing in sfeats_df should be counted__
__accurately in the head/tail of site_cat_matrix___
__sfeats_df[cat_code]______________________________


,cat_code
0,mp
1,mp
2,mp
3,re
4,wm
...,...
1601,wm
1602,mi
1603,mp
1604,wm


───────────────────────────────────────────────────

site_cat_matrix head__


,mp,re,wm,ot,mi,mt,site_cat_sum
site_id,,,,,,,
0100041,1,0,0,0,0,0,1
0100108,1,0,0,0,0,0,1
0100121,1,0,0,0,0,0,1
0100124,0,1,0,0,0,0,1
0100125,0,0,1,0,0,0,1
0100156,1,0,0,0,0,0,1
0100180,0,0,1,0,0,0,1
0100201,0,0,1,0,0,0,1
0100250,1,0,0,0,0,0,1


───────────────────────────────────────────────────

site_cat_matrix tail__


,mp,re,wm,ot,mi,mt,site_cat_sum
site_id,,,,,,,
1001455,0,0,0,1,0,0,1
1001508,0,0,0,0,1,0,1
1001733,0,0,0,1,0,0,1
1001761,0,0,0,1,0,0,1
1001865,0,0,0,0,1,0,1
1002020,0,0,0,1,0,0,1
1002095,1,0,0,0,0,0,1
1002155,0,0,0,1,0,0,1
1002171,0,0,0,1,0,0,1


───────────────────────────────────────────────────

__sfeats_df category matrix values should be same__
__as the category matrix values in site_cat_matrix_
__sfeats_df[cat_code]______________________________


,site_id,mp,re,wm,ot,mi,mt,site_cat_sum
0,0100041,1,0,0,0,0,0,1
1,0100108,1,0,0,0,0,0,1
2,0100121,1,0,0,0,0,0,1
3,0100124,0,1,0,0,0,0,1
4,0100125,0,0,1,0,0,0,1
5,0100156,1,0,0,0,0,0,1
6,0100180,0,0,1,0,0,0,1
7,0100201,0,0,1,0,0,0,1
8,0100250,1,0,0,0,0,0,1
9,0100251,0,0,1,0,0,0,1


,site_id,mp,re,wm,ot,mi,mt,site_cat_sum
1591,1001455,0,0,0,1,0,0,1
1592,1001508,0,0,0,0,1,0,1
1593,1001733,0,0,0,1,0,0,1
1594,1001761,0,0,0,1,0,0,1
1595,1001865,0,0,0,0,1,0,1
1596,1002020,0,0,0,1,0,0,1
1597,1002095,1,0,0,0,0,0,1
1598,1002155,0,0,0,1,0,0,1
1599,1002171,0,0,0,1,0,0,1
1600,1002174,0,0,1,0,0,0,1


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 4.2: ['cas_number'] MATRIX EXPLOSION
# ──────────────────────────────────────────────────────────────────────────────

# ──────────────────────────────────────────────────────────────────────────────
# >>> Create functions to clean and parse string lists                       <<<
# ──────────────────────────────────────────────────────────────────────────────

# >>> Strip extra whitespaces and return an organized list of cas_numbers <<<
def parse_cas_string_fixed(s):
    if pd.isna(s):
        return []
    return [i.strip(" '\"") for i in str(s).strip('[]').split(',')]

# >>> return cas_numbers fitting the the expected format of cas_numbers <<<
def is_valid_cas(s):
    return bool(re.fullmatch(r'\d{2,7}-\d{2}-\d', s))

# >>> return the value UNASSIGNED for entries that do not fit expected format <<<
def clean_cas_list(raw_list):
    return [s if is_valid_cas(s) else "UNASSIGNED" for s in raw_list]

# ──────────────────────────────────────────────────────────────────────────────
# >>> Make the Explosions happen                                             <<<
# ──────────────────────────────────────────────────────────────────────────────

# Enter long format.
cas_long = sfeats_df[['site_id', 'cas_number']].copy()
cas_long['site_id'] = cas_long['site_id'].astype(str)

# Application of parsing and cleaning functions.
cas_long['cas_list'] = cas_long['cas_number'].apply(parse_cas_string_fixed)
cas_long = cas_long.explode('cas_list').dropna()
cas_long['cas_clean'] = cas_long['cas_list'].apply(lambda s: s if is_valid_cas(s) else "UNASSIGNED")

# ──────────────────────────────────────────────────────────────────────────────
# >>> Make the EXPLOSION go and happen.                                      <<<
# ──────────────────────────────────────────────────────────────────────────────
cas_long['presence'] = 1
cas_matrix_binary = (
    cas_long.pivot_table(index='site_id', columns='cas_clean', values='presence', aggfunc='max', fill_value=0)
)
cas_matrix_binary['cas_sum'] = cas_matrix_binary.sum(axis=1)

# >>> Do a big ol' merge back into sfeats_df. <<<
common_cas = cas_matrix_binary.columns.drop('cas_sum').tolist()
cas_subset = cas_matrix_binary[common_cas + ['cas_sum']]
sfeats_df = sfeats_df.merge(cas_subset.reset_index(), on='site_id', how='left')
sfeats_df = sfeats_df.drop(columns=['cas_number'])

# ──────────────────────────────────────────────────────────────────────────────
# >>> VALIDATION OUTPUT                                                      <<<
# ──────────────────────────────────────────────────────────────────────────────
print('__sfeats_df should have 0 NAs__')
display(sfeats_df.isna().sum())
print('───────────────────────────────────────────────────')
print()
print('__sfeats_df should have 1606 observations, 740 columns__')
display(sfeats_df.info())
print('───────────────────────────────────────────────────')
print()
print('__sfeats_df head and tail should be as expected__')
print('__only three non-numeric columns left to handle__')
display(sfeats_df.head())
print('───────────────────────────────────────────────────')
print()
print('__check pre and post of column titles - no garbage cas_numbers_______')
print('__are to be permitted as column headers; they will be last if there__')
display(sfeats_df.columns[:15])
print()
display(sfeats_df.columns[-15:])
print('───────────────────────────────────────────────────')

__sfeats_df should have 0 NAs__


,0
site_id,0
lat,0
lon,0
fed_facility,0
native_land,0
...,...
99-87-6,0
99-99-0,0
993-13-5,0
UNASSIGNED,0


───────────────────────────────────────────────────

__sfeats_df should have 1606 observations, 740 columns__
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1606 entries, 0 to 1605
Columns: 739 entries, site_id to cas_sum
dtypes: float64(5), int64(729), object(5)
memory usage: 9.1+ MB


None

───────────────────────────────────────────────────

__sfeats_df head and tail should be as expected__
__only three non-numeric columns left to handle__


,site_id,lat,lon,fed_facility,native_land,npl_status,hazard_rank_score,final_list_days,proposed_days,proposal_gulf,...,98-95-3,99-08-1,99-09-2,99-35-4,99-65-0,99-87-6,99-99-0,993-13-5,UNASSIGNED,cas_sum
0,0100041,41.940561,-71.966939,n,n,currently on the final npl,33.71,12733.0,13340,607.0,...,0,0,0,0,0,0,0,0,0,35
1,0100108,41.481110,-72.681388,n,n,currently on the final npl,33.94,12873.0,13340,467.0,...,0,0,0,0,0,0,0,0,1,43
2,0100121,41.708331,-71.829169,n,n,deleted from the final npl,41.06,13678.0,14085,407.0,...,0,0,0,0,0,0,0,0,0,30
3,0100124,41.619600,-72.878000,n,n,currently on the final npl,44.93,15091.0,15343,252.0,...,0,0,0,0,0,0,0,0,1,94
4,0100125,41.669450,-71.964161,n,n,currently on the final npl,36.72,15091.0,15343,252.0,...,0,0,0,0,0,0,0,0,1,66


───────────────────────────────────────────────────

__check pre and post of column titles - no garbage cas_numbers_______
__are to be permitted as column headers; they will be last if there__


Index(['site_id', 'lat', 'lon', 'fed_facility', 'native_land', 'npl_status',
       'hazard_rank_score', 'final_list_days', 'proposed_days',
       'proposal_gulf', 'is_finalized', 'is_deleted', 'fips', 'mp', 're'],
      dtype='object')

Index(['98-06-6', '98-82-8', '98-86-2', '98-87-3', '98-88-4', '98-95-3',
       '99-08-1', '99-09-2', '99-35-4', '99-65-0', '99-87-6', '99-99-0',
       '993-13-5', 'UNASSIGNED', 'cas_sum'],
      dtype='object')

───────────────────────────────────────────────────


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 4.3: Encoding remaining non-numeric column: ['npl_status']
# ──────────────────────────────────────────────────────────────────────────────
npl_remap = {'currently on the final npl': 'current',
             'deleted from the final npl': 'deleted',
             'proposed for npl': 'proposed'}
sfeats_df['npl_status'] = sfeats_df['npl_status'].map(npl_remap).fillna('other')
sfeats_df = pd.get_dummies(sfeats_df, columns=['npl_status'], prefix='npl')

# ──────────────────────────────────────────────────────────────────────────────
# >>> VALIDATION OUTPUT                                                      <<<
# ──────────────────────────────────────────────────────────────────────────────
npl_counts = [
    sfeats_df['npl_current'].sum(),
    sfeats_df['npl_deleted'].sum(),
    sfeats_df['npl_proposed'].sum()
]
npl_rows = [
    ["Current",  "1232", int(npl_counts[0])],
    ["Deleted",  "0345", int(npl_counts[1])],
    ["Proposed", "0029", int(npl_counts[2])]
]
print('Validate expectations for new column values:')
print(tabulate(npl_rows, headers=["NPL Status", "Expected", "npl_encoding values"],
               tablefmt="grid"))

Validate expectations for new column values:
+--------------+------------+-----------------------+
| NPL Status   |   Expected |   npl_encoding values |
+==============+============+=======================+
| Current      |       1232 |                  1232 |
+--------------+------------+-----------------------+
| Deleted      |       0345 |                   345 |
+--------------+------------+-----------------------+
| Proposed     |       0029 |                    29 |
+--------------+------------+-----------------------+


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 4.4: Encoding remaining non-numeric columns:
#           ['fed_facility', 'native_land']
# ──────────────────────────────────────────────────────────────────────────────
ff_remap = {'n': 0,
            'y': 1}
sfeats_df['fed_facility'] = sfeats_df['fed_facility'].map(ff_remap).fillna(-1)

nat_land_remap = {'n': 0,
                  'y': 1}
sfeats_df['native_land'] = sfeats_df['native_land'].map(nat_land_remap).fillna(-1)


# ──────────────────────────────────────────────────────────────────────────────
# >>> VALIDATION OUTPUT - fed_facility == 170/1436, native_land = 169/1437   <<<
# ──────────────────────────────────────────────────────────────────────────────
npl_counts = [
    sfeats_df['fed_facility'].sum(),
    sfeats_df['native_land'].sum()
]
npl_rows = [
    ["fed_facility",  "0170", int(npl_counts[0])],
    ["native_land",  "0169", int(npl_counts[1])]
]
print('Validate expectations for new column values:')
print(tabulate(npl_rows, headers=["column", "expected", "true_count"],
               tablefmt="grid"))

# ──────────────────────────────────────────────────────────────────────────────
# >>> STRUCTURAL NAs - These NAs indicate a site is proposed but not listed. <<<
# ──────────────────────────────────────────────────────────────────────────────
sfeats_df['final_list_days'] = sfeats_df['final_list_days'].fillna(-1)
sfeats_df['proposal_gulf'] = sfeats_df['proposal_gulf'].fillna(-1)

# ──────────────────────────────────────────────────────────────────────────────
# >>> VALIDATION OUTPUT - fed_facility == 170/1436, native_land = 169/1437   <<<
# ──────────────────────────────────────────────────────────────────────────────
print('Sum of values == -1 in final_list_days and proposal_gulf.')
print('Should equal 31 for both.')
display((sfeats_df['final_list_days'] == -1).sum())
display((sfeats_df['proposal_gulf'] == -1).sum())
print()
display(sfeats_df.info())
print()
display(sfeats_df.head())
print()
display(sfeats_df.tail())

Validate expectations for new column values:
+--------------+------------+--------------+
| column       |   expected |   true_count |
+==============+============+==============+
| fed_facility |       0170 |          170 |
+--------------+------------+--------------+
| native_land  |       0169 |          169 |
+--------------+------------+--------------+
Sum of values == -1 in final_list_days and proposal_gulf.
Should equal 31 for both.


31

31


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1606 entries, 0 to 1605
Columns: 741 entries, site_id to npl_proposed
dtypes: bool(3), float64(5), int64(731), object(2)
memory usage: 9.0+ MB


None

,site_id,lat,lon,fed_facility,native_land,hazard_rank_score,final_list_days,proposed_days,proposal_gulf,is_finalized,...,99-35-4,99-65-0,99-87-6,99-99-0,993-13-5,UNASSIGNED,cas_sum,npl_current,npl_deleted,npl_proposed
0,0100041,41.940561,-71.966939,0,0,33.71,12733.0,13340,607.0,1,...,0,0,0,0,0,0,35,True,False,False
1,0100108,41.481110,-72.681388,0,0,33.94,12873.0,13340,467.0,1,...,0,0,0,0,0,1,43,True,False,False
2,0100121,41.708331,-71.829169,0,0,41.06,13678.0,14085,407.0,1,...,0,0,0,0,0,0,30,False,True,False
3,0100124,41.619600,-72.878000,0,0,44.93,15091.0,15343,252.0,1,...,0,0,0,0,0,1,94,True,False,False
4,0100125,41.669450,-71.964161,0,0,36.72,15091.0,15343,252.0,1,...,0,0,0,0,0,1,66,True,False,False


,site_id,lat,lon,fed_facility,native_land,hazard_rank_score,final_list_days,proposed_days,proposal_gulf,is_finalized,...,99-35-4,99-65-0,99-87-6,99-99-0,993-13-5,UNASSIGNED,cas_sum,npl_current,npl_deleted,npl_proposed
1601,1002476,42.264800,-121.746500,0,1,0.0,4856.0,5046,190.0,1,...,0,0,0,0,0,0,3,True,False,False
1602,1002616,42.854472,-123.382611,0,1,50.0,6314.0,6510,196.0,1,...,0,0,0,0,0,1,5,True,False,False
1603,1002655,47.583889,-122.362500,0,1,50.0,6510.0,6671,161.0,1,...,0,0,0,0,0,1,28,True,False,False
1604,1002857,48.388353,-124.656711,0,1,50.0,4038.0,4240,202.0,1,...,0,0,0,0,0,0,1,True,False,False
1605,1002907,47.578544,-122.642136,0,1,50.0,4619.0,4856,237.0,1,...,0,0,0,0,0,0,1,True,False,False


```
# ───────────────────────────────────────────────────────────────────────
# CELL 5.0: FREEZE CLEANED AND FINALIZED SUPERFUND FEATURE DATAFRAME
#           CREATE AND FREEZE FINALIZED FEATURE MATRIX  
# ───────────────────────────────────────────────────────────────────────
```

In [ ]:
# ───────────────────────────────────────────────────────────────────────
# CELL 5.1: FREEZE CLEANED AND FINALIZED SUPERFUND FEATURE DATAFRAME
# ───────────────────────────────────────────────────────────────────────
superfund_feature_dataframe = sfeats_df.copy(deep=True)
superfund_feature_matrix = sfeats_df.drop(columns = 'site_id').copy(deep=True)


superfund_feature_dataframe.to_csv('DATA001_superfund_feature_dataframe_v1.csv', index=False, float_format="%.10g")
superfund_feature_matrix.to_csv('DATA001_superfund_feature_matrix_v1.csv', index=False, float_format="%.10g")


In [ ]:
# ───────────────────────────────────────────────────────────────────────
# CELL 5.2: LOG MAP OF SFEATS_DF INDEX (SITE_ID -> NODES)
#           TEST AND RECORD FINAL MATRIX SHAPE
# ───────────────────────────────────────────────────────────────────────
data100_node_id_map = sfeats_df[['site_id']].reset_index().rename(columns={'index': 'row_index'})
data100_node_id_map.to_csv('DATA001_superfund_node_id_map_v1.csv', index=False, float_format="%.10g")

# ───────────────────────────────────────────────────────────────────────
# VALIDATION OUTPUT: SFEATS_DF INDEX NODE MAP (SITE_ID -> NODES)
#                    DATA COLUMNS SHOULD MATCH IN BELOW
# ───────────────────────────────────────────────────────────────────────

# >>> Compare tops and bottoms.
head_compare = pd.DataFrame({
    'sfeats_df_head': sfeats_df['site_id'].head().values,
    'node_id_map_head': data100_node_id_map['site_id'].head().values
})

# Tail comparison
tail_compare = pd.DataFrame({
    'sfeats_df_tail': sfeats_df['site_id'].tail().values,
    'node_id_map_tail': data100_node_id_map['site_id'].tail().values
})

print("── HEAD COMPARISON ──")
display(head_compare)

print("── TAIL COMPARISON ──")
display(tail_compare)

# ───────────────────────────────────────────────────────────────────────
# CELL 5.2: LOG FINAL FEATURE MATRIX FEATURES COUNT
# ───────────────────────────────────────────────────────────────────────
print(f"Final matrix shape: {superfund_feature_matrix.shape}")

── HEAD COMPARISON ──


,sfeats_df_head,node_id_map_head
0,0100041,0100041
1,0100108,0100108
2,0100121,0100121
3,0100124,0100124
4,0100125,0100125


── TAIL COMPARISON ──


,sfeats_df_tail,node_id_map_tail
0,1002476,1002476
1,1002616,1002616
2,1002655,1002655
3,1002857,1002857
4,1002907,1002907


Final matrix shape: (1606, 740)


In [ ]:
# ───────────────────────────────────────────────────────────────────────
# CELL 999.0: SCRATCH CELL
# ───────────────────────────────────────────────────────────────────────